In [1]:
import pandas as pd

# Replace 'your_file.csv' with the actual file name
data = pd.read_csv('D:/sathya/project/malicious_phish.csv')

# Display the first few rows of the dataframe to verify the content
print(data.head())

# Classify unique values in the 'type' column
unique_types = data['type'].unique()

# Create a new DataFrame with classified unique values
cd = pd.DataFrame(unique_types, columns=['Unique Types'])

# Display the unique types
print(cd)


                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
  Unique Types
0     phishing
1       benign
2   defacement
3      malware


#preprocessing

In [2]:
import pandas as pd

# Load the CSV file
data = pd.read_csv('D:/sathya/project/malicious_phish.csv')

# Inspect the first few rows
print(data.head())


                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [5]:
from statsmodels.sandbox.regression.sympy_diff import df
import re

def tokenize_url(url):
    # Use regex to split by non-alphanumeric characters
    tokens= re.split(r'\W+', url)
    # Remove empty tokens
    tokens = [token for token in tokens if token]
    return tokens

# Apply the tokenization function to the URL column
df['tokens'] = df['url'].apply(tokenize_url)


In [6]:
from collections import Counter

# Flatten the list of token lists and create a Counter object
all_tokens = [token for tokens in df['tokens'] for token in tokens]
vocabulary = Counter(all_tokens)

# Optional: Set a maximum vocabulary size to limit the number of tokens
max_vocab_size = 5000
vocab = {token: idx for idx, (token, _) in enumerate(vocabulary.most_common(max_vocab_size), 1)}


In [7]:
def tokens_to_indices(tokens, vcab):
    return [vcab.get(token, 0) for token in tokens]  # Use 0 for unknown tokens

df['token_indices'] = df['tokens'].apply(lambda tokens: tokens_to_indices(tokens, vocab))


In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define the maximum sequence length
max_seq_length = 100

# Pad the sequences with 0s
padded_sequences = pad_sequenxes(df['token_indices'], maxlen=max_seq_length, padding='post', truncating='post')

# Convert to a DataFrame for inspection
padded_df = pd.DataFrame(padded_sequences)
print(padded_df.head())


     0     1     2     3    4   5   6    7   8   9   ...  90  91  92  93  94  \
0    53  3826     1    53    0   0   0    0   0   0  ...   0   0   0   0   0   
1  1857     1   121     0    4   0   0    0   0   0  ...   0   0   0   0   0   
2     0     7     0  2794   15  14   0    0   0   0  ...   0   0   0   0   0   
3     2     3  3175     0  228   6   5    9  18  11  ...   0   0   0   0   0   
4     2  2309  3465    10    6   5   9  251  41  23  ...   0   0   0   0   0   

   95  96  97  98  99  
0   0   0   0   0   0  
1   0   0   0   0   0  
2   0   0   0   0   0  
3   0   0   0   0   0  
4   0   0   0   0   0  

[5 rows x 100 columns]


In [9]:
# Map labels to numerical values
label_mapping = {'phishing': 0, 'benign': 1, 'defacement': 2, 'malware': 3}
df['label'] = df['type'].map(label_mapping)

# Separate features and labels
X = padded_sequences
y = df['label'].values


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#Model

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

# Define the model
model = Sequential([
    Embedding(input_dim=max_vocab_size + 1, output_dim=128),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')  # 4 classes: phishing, benign, defacement, malware
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



In [12]:
# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

Epoch 1/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 186s 25ms/step - accuracy: 0.9270 - loss: 0.2030 - val_accuracy: 0.9543 - val_loss: 0.1245
Epoch 2/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 181s 25ms/step - accuracy: 0.9585 - loss: 0.1109 - val_accuracy: 0.9574 - val_loss: 0.1161
Epoch 3/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 403s 55ms/step - accuracy: 0.9626 - loss: 0.0989 - val_accuracy: 0.9587 - val_loss: 0.1132
Epoch 4/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 387s 53ms/step - accuracy: 0.9645 - loss: 0.0938 - val_accuracy: 0.9575 - val_loss: 0.1172
Epoch 5/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 315s 43ms/step - accuracy: 0.9660 - loss: 0.0883 - val_accuracy: 0.9585 - val_loss: 0.1162


In [13]:
 # Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy:.4f}')


4070/4070 ━━━━━━━━━━━━━━━━━━━━ 57s 14ms/step - accuracy: 0.9587 - loss: 0.1175
Test Accuracy: 0.9585


In [14]:
# Predict classes for the test set
y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=-1)


4070/4070 ━━━━━━━━━━━━━━━━━━━━ 57s 14ms/step


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

# Compute the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)

# Print the confusion matrix
print('Confusion Matrix:')
print(conf_matrix)

# Print the classification report
print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))


Confusion Matrix:
[[14902  3718   162    54]
 [  983 84748    28    19]
 [  149    13 18934     8]
 [  120   124    27  6250]]
Classification Report:
              precision    recall  f1-score   support

    phishing       0.92      0.79      0.85     18836
      benign       0.96      0.99      0.97     85778
  defacement       0.99      0.99      0.99     19104
     malware       0.99      0.96      0.97      6521

    accuracy                           0.96    130239
   macro avg       0.96      0.93      0.95    130239
weighted avg       0.96      0.96      0.96    130239



In [16]:
import numpy as np
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print('Confusion Matrix:')
print(conf_matrix)

print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))

# Calculate accuracy from confusion matrix
correct_predictions = np.trace(conf_matrix)
total_predictions = np.sum(conf_matrix)
accuracy = correct_predictions / total_predictions
print(f'Calculated Accuracy from Confusion Matrix: {accuracy:.4f}')

Confusion Matrix:
[[14902  3718   162    54]
 [  983 84748    28    19]
 [  149    13 18934     8]
 [  120   124    27  6250]]
Classification Report:
              precision    recall  f1-score   support

    phishing       0.92      0.79      0.85     18836
      benign       0.96      0.99      0.97     85778
  defacement       0.99      0.99      0.99     19104
     malware       0.99      0.96      0.97      6521

    accuracy                           0.96    130239
   macro avg       0.96      0.93      0.95    130239
weighted avg       0.96      0.96      0.96    130239

Calculated Accuracy from Confusion Matrix: 0.9585
